# 🖼️ Image Classification using CNN
### Transfer Learning with ResNet50
**Dataset:** Intel Image Classification (Kaggle)
**Classes:** buildings | forest | glacier | mountain | sea | street

## 📦 Step 1 — Install Dependencies

In [ ]:
!pip install kaggle -q
import os, numpy as np, matplotlib.pyplot as plt, seaborn as sns
from sklearn.metrics import classification_report, confusion_matrix
import tensorflow as tf
from tensorflow.keras import layers, models, optimizers, callbacks
from tensorflow.keras.applications import ResNet50
from tensorflow.keras.preprocessing.image import ImageDataGenerator
import warnings; warnings.filterwarnings('ignore')
print("TensorFlow:", tf.__version__)

## 🔑 Step 2 — Upload kaggle.json API Key

In [ ]:
from google.colab import files
files.upload()   # upload your kaggle.json here

os.makedirs('/root/.kaggle', exist_ok=True)
os.rename('kaggle.json', '/root/.kaggle/kaggle.json')
os.chmod('/root/.kaggle/kaggle.json', 0o600)
print("✅ Kaggle API key configured!")

## 📥 Step 3 — Download Dataset

In [ ]:
!kaggle datasets download -d puneet6060/intel-image-classification -p /content/
!unzip -q /content/intel-image-classification.zip -d /content/data/
!ls /content/data/

## ⚙️ Step 4 — Configuration

In [ ]:
TRAIN_DIR = '/content/data/seg_train/seg_train'
TEST_DIR  = '/content/data/seg_test/seg_test'
IMG_SIZE  = 150; BATCH_SIZE = 32; EPOCHS = 25; LEARNING_RATE = 1e-4; SEED = 42
CLASS_NAMES = ['buildings', 'forest', 'glacier', 'mountain', 'sea', 'street']
tf.random.set_seed(SEED); np.random.seed(SEED)
print("Train classes:", sorted(os.listdir(TRAIN_DIR)))

## 📁 Step 5 — Data Generators + Augmentation

In [ ]:
train_datagen = ImageDataGenerator(rescale=1/255, rotation_range=20,
    width_shift_range=0.15, height_shift_range=0.15, shear_range=0.1,
    zoom_range=0.15, horizontal_flip=True, fill_mode='nearest', validation_split=0.1)
test_datagen = ImageDataGenerator(rescale=1/255)

train_gen = train_datagen.flow_from_directory(TRAIN_DIR, target_size=(IMG_SIZE,IMG_SIZE),
    batch_size=BATCH_SIZE, class_mode='categorical', subset='training', seed=SEED)
val_gen   = train_datagen.flow_from_directory(TRAIN_DIR, target_size=(IMG_SIZE,IMG_SIZE),
    batch_size=BATCH_SIZE, class_mode='categorical', subset='validation', seed=SEED)
test_gen  = test_datagen.flow_from_directory(TEST_DIR, target_size=(IMG_SIZE,IMG_SIZE),
    batch_size=BATCH_SIZE, class_mode='categorical', shuffle=False)

## 🔍 Step 6 — Visualize Samples

In [ ]:
X_b, y_b = next(train_gen)
labels = list(train_gen.class_indices.keys())
fig, axes = plt.subplots(2, 6, figsize=(18, 7))
for i, ax in enumerate(axes.flat):
    ax.imshow(X_b[i]); ax.set_title(labels[np.argmax(y_b[i])], fontsize=9); ax.axis('off')
plt.suptitle('Sample Training Images — Intel Dataset', fontsize=13)
plt.tight_layout(); plt.show()

## 🏗️ Step 7 — Build Model (ResNet50 + Custom Head)

In [ ]:
base = ResNet50(weights='imagenet', include_top=False, input_shape=(IMG_SIZE,IMG_SIZE,3))
for layer in base.layers[:-20]: layer.trainable = False

model = models.Sequential([
    base,
    layers.GlobalAveragePooling2D(),
    layers.Dense(512, activation='relu'),
    layers.BatchNormalization(),
    layers.Dropout(0.4),
    layers.Dense(128, activation='relu'),
    layers.Dropout(0.3),
    layers.Dense(6, activation='softmax')
])
model.compile(optimizer=optimizers.Adam(LEARNING_RATE),
              loss='categorical_crossentropy', metrics=['accuracy'])
model.summary()

## 🚀 Step 8 — Train

In [ ]:
cb_list = [
    callbacks.EarlyStopping(patience=6, restore_best_weights=True, verbose=1),
    callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=3, verbose=1),
    callbacks.ModelCheckpoint('/content/imgcls_best.h5', save_best_only=True, verbose=1)
]
history = model.fit(train_gen, validation_data=val_gen, epochs=EPOCHS, callbacks=cb_list)

## 📈 Step 9 — Training Curves

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].plot(history.history['accuracy'], label='Train')
axes[0].plot(history.history['val_accuracy'], label='Validation')
axes[0].set_title('Accuracy'); axes[0].set_xlabel('Epoch'); axes[0].legend()
axes[1].plot(history.history['loss'], label='Train')
axes[1].plot(history.history['val_loss'], label='Validation')
axes[1].set_title('Loss'); axes[1].set_xlabel('Epoch'); axes[1].legend()
plt.suptitle('Image Classification — Training History', fontsize=14)
plt.tight_layout(); plt.show()

## 🧪 Step 10 — Evaluate

In [ ]:
loss, acc = model.evaluate(test_gen, verbose=0)
print(f"\n✅ Test Accuracy: {acc*100:.2f}%  |  Test Loss: {loss:.4f}")

test_gen.reset()
y_pred = np.argmax(model.predict(test_gen), axis=1)
y_true = test_gen.classes
print("\n📊 Classification Report:")
print(classification_report(y_true, y_pred, target_names=labels))

In [ ]:
cm = confusion_matrix(y_true, y_pred)
plt.figure(figsize=(10, 8))
sns.heatmap(cm, annot=True, fmt='d', cmap='Greens',
            xticklabels=labels, yticklabels=labels)
plt.title('Confusion Matrix — Image Classification', fontsize=14)
plt.ylabel('True Label'); plt.xlabel('Predicted Label')
plt.tight_layout(); plt.show()

## 🖼️ Step 11 — Sample Predictions

In [ ]:
test_gen.reset()
X_b, y_b = next(test_gen)
preds = np.argmax(model.predict(X_b), axis=1)
fig, axes = plt.subplots(3, 5, figsize=(18, 10))
for i, ax in enumerate(axes.flat):
    if i >= len(X_b): break
    ax.imshow(X_b[i])
    t = labels[np.argmax(y_b[i])]; p = labels[preds[i]]
    ax.set_title(f'T: {t}\nP: {p}', color='green' if t==p else 'red', fontsize=8)
    ax.axis('off')
plt.suptitle('Sample Predictions', fontsize=13); plt.tight_layout(); plt.show()

In [ ]:
model.save('/content/image_classification_final.h5')
print("✅ Model saved to /content/image_classification_final.h5")